# Level-by-level agents on a Flink session cluster

Each cell submits one Flink job into a long-running session cluster (JobManager on `localhost:8081`).
All five levels coexist; cross-level hops travel over **ZeroMQ PUSH/PULL** (no Kafka/Fluss topic
burned for transient data). Fluss holds the durable boundaries: raw Coinbase ingest at L0 and
the final adjudicated alerts at L5.

Chain (submit in this order — the PULL side of each hop must bind first):

```
L5  (binds 5564, also binds control 5559) → Fluss alerts
L4  (binds 5563)                           → ZMQ 5564
L3  (binds 5562)                           → ZMQ 5563
L2  (binds 5561)                           → ZMQ 5562
L1  (binds 5560)                           → ZMQ 5561
producer (Coinbase WS)                     → ZMQ 5560
```

Once all six are up, flip debug on L5 from this notebook via the `DebugFlipper` — debug events
stream back to a sink you tail in the last cell.

**Prereqs:**
- `mvn -DskipTests package` (jar)
- `examples-bin/run-session-cluster.sh` (Fluss + Flink session cluster)
- `.env` with `ANTHROPIC_API_KEY=sk-ant-...` (lights up the L5 LLM tier)
- `pip install requests pyzmq python-dotenv` (already in `.venv`)

## 1. Bootstrap — load `.env`, locate jar, import session client

In [1]:
import os, pathlib, sys
repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
from dotenv import load_dotenv  # required — pip install python-dotenv
_env_path = repo_root / '.env'
_loaded = load_dotenv(_env_path, override=True)
print(f'.env loaded={_loaded} from {_env_path} (exists={_env_path.exists()})')

_py_src = str(repo_root / 'python')
if _py_src not in sys.path:
    sys.path.insert(0, _py_src)

# sanity check: bail with a clear message if the kernel lacks deps.
_missing = []
for _m in ('requests', 'zmq'):
    try:
        __import__(_m)
    except ModuleNotFoundError:
        _missing.append('pyzmq' if _m == 'zmq' else _m)
if _missing:
    raise SystemExit(
        f'Missing modules {_missing} in this kernel ({sys.executable}). '
        f'pip install ' + ' '.join(_missing)
    )

from agentic_flink.session import SessionClient, DebugFlipper, operator_ids
client = SessionClient('http://localhost:8081')

jars = sorted((repo_root/'target').glob('agentic-flink-*.jar'))
jars = [j for j in jars if 'original' not in j.name and 'sources' not in j.name]
assert jars, 'build the jar first: mvn -DskipTests package'
jar_path = jars[-1]

key = os.environ.get('ANTHROPIC_API_KEY') or ''
print('interpreter:', sys.executable)
print('jar:', jar_path)
print('LLM tier on L5:', 'Claude' if key.startswith('sk-ant-') else 'disabled')

.env loaded=True from /Users/bengamble/Agentic-Flink/.env (exists=True)
interpreter: /private/tmp/af-venv/bin/python3
jar: /Users/bengamble/Agentic-Flink/target/agentic-flink-1.0.0-SNAPSHOT.jar
LLM tier on L5: Claude


## 2. Bring up the cluster

If the session cluster isn't already running, open a terminal and run:

```
podman network create agentic-flink-network   # once
bash examples-bin/run-session-cluster.sh
```

Then re-run the cell below — it polls `:8081/overview` until the cluster answers.

In [2]:
overview = client.wait_ready(timeout_s=90)
print('cluster:', {k: overview.get(k) for k in ('taskmanagers','slots-total','slots-available','jobs-running','flink-version')})

KeyboardInterrupt: 

## 3. Upload the jar (once per session)

If the jar's already there, `latest_jar` returns its id; otherwise we POST it.

In [ ]:
jar_id = client.latest_jar('agentic-flink')
if jar_id is None:
    jar_id = client.upload_jar(jar_path)
    print('uploaded:', jar_id)
else:
    print('reusing uploaded jar:', jar_id)

## 4. Submit Level 5 first (binds the downstream-most ports)

L5 binds **5564** (data PULL) and **5559** (control PULL). It writes adjudicated alerts to the
Fluss table `agentic.alerts` and taps the debug side-output to ZMQ **5558** for live tailing
from this notebook.

In [ ]:
l5 = client.submit_level(
    jar_id,
    '5',
    in_endpoint='tcp://0.0.0.0:5564',
    out_endpoint='fluss://agentic.alerts',
    control_endpoint='tcp://0.0.0.0:5559',
    debug_sink_endpoint='tcp://0.0.0.0:5558',
    anthropic_key=os.environ.get('ANTHROPIC_API_KEY') or None,
    extra_args={'fluss-bootstrap': 'fluss-coordinator:9120'},
    name='L5-market-agent',
)
print('L5 job:', l5.job_id)

## 5. Levels 4 → 3 → 2 → 1

Each binds its PULL port (so it must exist before upstream connects).

In [ ]:
l4 = client.submit_level(jar_id, '4', in_endpoint='tcp://0.0.0.0:5563', out_endpoint='tcp://flink-taskmanager:5564', window_ms=5000, top_n=5, name='L4-features')
l3 = client.submit_level(jar_id, '3', in_endpoint='tcp://0.0.0.0:5562', out_endpoint='tcp://flink-taskmanager:5563', top_n=5, name='L3-topN')
l2 = client.submit_level(jar_id, '2', in_endpoint='tcp://0.0.0.0:5561', out_endpoint='tcp://flink-taskmanager:5562', name='L2-enrich')
l1 = client.submit_level(jar_id, '1', in_endpoint='tcp://0.0.0.0:5560', out_endpoint='tcp://flink-taskmanager:5561', name='L1-passthrough')
print('submitted L4-L1:', l4.job_id, l3.job_id, l2.job_id, l1.job_id)

## 6. Producer — Coinbase WebSocket → ZMQ 5560

The producer connects (not binds) to L1's PULL, so L1 must already be up.

In [ ]:
prod = client.submit_level(
    jar_id,
    'producer',
    out_endpoint='tcp://flink-taskmanager:5560',
    products=[s.strip() for s in os.environ.get('COINBASE_PRODUCTS', 'BTC-USD,ETH-USD,SOL-USD').split(',')],
    name='L0-coinbase-producer',
)
print('producer job:', prod.job_id)

## 7. Confirm — six jobs running

In [ ]:
for j in client.jobs():
    print(f"{j.get('state'):<10} {j.get('jid')[:8]}  {j.get('name')}")

print()
print('Open the UI: http://localhost:8081 — you should see all six DAGs.')

## 8. Discover operator ids for the control plane

Every framework operator names itself with a stable id so the broadcast control plane can
target it. The id for the L5 agentic operator is `L5.market-agent`.

In [ ]:
print('L5 operator ids:')
for op in operator_ids(l5.plan()):
    print(' -', op)

## 9. Flip debug on the L5 operator (default TTL: 2 minutes)

Push a `DebugControl` directly into L5's control PULL on `:5559`. After ~120 seconds the
directive auto-expires; pin it with `.pinned(...)` to persist, cancel with `.off(...)`.

In [ ]:
flipper = DebugFlipper('tcp://localhost:5559')
flipper.on('L5.market-agent', ttl_seconds=300)   # 5 min — plenty of time to watch
print('debug flipped ON for L5.market-agent')

## 10. Tail the debug stream (Ctrl-C / interrupt to stop)

L5 sinks its debug side-output to `tcp://0.0.0.0:5558` (PUSH). Pull from it here.

In [ ]:
import json, zmq
ctx = zmq.Context.instance()
tail = ctx.socket(zmq.SUB)
tail.connect('tcp://localhost:5558')  # L5 PUB binds inside the container; we SUB-connect
tail.setsockopt(zmq.SUBSCRIBE, b'')
tail.setsockopt(zmq.RCVTIMEO, 1000)

MAX_EVENTS = 20
received = 0
try:
    while received < MAX_EVENTS:
        try:
            raw = tail.recv()
        except zmq.Again:
            continue
        evt = json.loads(raw)
        received += 1
        payload = evt.get('payload', {})
        print(
            f"{evt['operatorId']:<18} {evt['kind']:<8} "
            f"instr={payload.get('instrumentId')} "
            f"spread={payload.get('spread', 0):.2f}bp "
            f"verdict={payload.get('verdict')} decidedBy={payload.get('decidedBy')} "
            f"risk={payload.get('combinedRisk', 0):.2f}"
        )
finally:
    tail.close()
print(f'\nreceived {received} debug events.')


## 11. Cleanup — cancel all six jobs

In [ ]:
for h in (prod, l1, l2, l3, l4, l5):
    try:
        h.cancel()
        print('cancelled:', h.job_id)
    except Exception as e:
        print('cancel error:', h.job_id, e)
flipper.close()